# step A-2 — 실제 저장소 코드 (RQ1 외적 타당성)

**대응 RQ:** RQ1 대조 — 합성이 아니라 **실제 파일**을 선행 코드로 써서, 언어 관습을 자연 실험으로 이용.

**언어 관습 2×2** (지침만 무작위, 실코드는 무조작):

| 선행 파일 | 지침=camel | 지침=snake |
|---|---|---|
| Python (자연 snake) | 충돌 | 일치 |
| JavaScript (자연 camel) | 일치 | 충돌 |

충돌 칸에서 준수율이 낮으면 → 모델이 지침보다 **언어 관습(형태)**을 따름(실코드에서도 형태 지배).
실파일 출처·라이선스: `data/repo_files/SOURCE.md` (CPython PSF, axios MIT). 설계: `docs/stepA-2/plan.md`.

> 생성률만 측정, 내부 개입 없음. 3B fp16, 재개 가능.

In [ ]:
# 환경 설정
!pip install -q transformers accelerate torch matplotlib pandas
import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
SEED=0; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin stepA-2/real-repo
!git checkout stepA-2/real-repo
!git pull --quiet origin stepA-2/real-repo
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# 조건 설정 — 언어 × 지침 × 실파일 (source=repo)
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Source, InstructionForm, Notation)
MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
STEP = 'stepA-2'
FILES = {
    'python':     ['python/fnmatch.py', 'python/textwrap.py', 'python/string.py'],
    'javascript': ['javascript/utils.js', 'javascript/buildURL.js', 'javascript/formDataToJSON.js'],
}
INSTR = [Notation.CAMEL, Notation.SNAKE]
def make(lang, f, target):
    return Condition(model=MODEL,
        preceding=PrecedingCode(n_compliant=0, source=Source.REPO, repo_lang=lang, repo_file=f),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=target), seed=0)
conditions = [make(lang, f, t) for lang, fs in FILES.items() for f in fs for t in INSTR]
PREDICTION = ('일치 칸(파일 관습=지침) 높음, 충돌 칸 낮음이면 언어 관습(형태)이 지침을 압도 → '
              'RQ1 외적 타당성. 특히 Python+camel(충돌)에서 바닥 예상(step A와 일치).')
print(len(conditions), '조건 =', '2 lang x 3 files x 2 instr')

In [ ]:
# 실행 — 조건별 순차 생성 + 즉시 저장(재개). 원문 자동 저장.
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
handle = load_model(MODEL)
print('layers:', handle.num_layers)
new = skipped = 0
for c in conditions:
    if result_path(c, step=STEP).exists():
        skipped += 1; continue
    out = run(c, handle=handle)
    save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                             step=STEP, rq='RQ1', prediction=PREDICTION))
    new += 1
    print(f'{c.preceding.repo_lang:10} {c.preceding.repo_file.split("/")[-1]:22} instr={c.instruction.target_notation.value:5} -> {out.metrics.extra["turn_notations"]}')
print(f'완료: 새로 {new}, 건너뜀 {skipped}')

In [ ]:
# 결과 로드
from harness import result_path
from harness.results import load_result
records = [load_result(result_path(c, step=STEP)) for c in conditions]
print('로드:', len(records), '건 → results/'+STEP+'/')

In [ ]:
# 요약 — 2x2 준수율 (언어 x 지침). 생성된 모든 함수(턴) 기준.
import pandas as pd, matplotlib.pyplot as plt
rows = []
for r in records:
    c = r.condition; tgt = c.instruction.target_notation.value
    for nt in r.metrics.extra['turn_notations']:
        rows.append({'lang': c.preceding.repo_lang, 'instr': tgt, 'compliant': nt == tgt})
df = pd.DataFrame(rows)
piv = df.groupby(['lang','instr'])['compliant'].mean().unstack().round(3)
print('준수율 2x2 (행=파일 언어, 열=지침 목표표기):'); print(piv)
print('\n자연 관습: python=snake, javascript=camel  →  대각선 반대편이 충돌 칸')
ax = piv.plot(kind='bar', figsize=(5,3))
ax.set_ylabel('준수율(생성)'); ax.set_ylim(0,1); ax.set_title('RQ1 외적타당성: 언어 x 지침')
ax.legend(title='지침'); plt.grid(axis='y', alpha=.3); plt.tight_layout(); plt.show()